# HW-A2: 實驗主程式
## 0. 環境設定與套件載入

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.utils import shuffle

# 確保 output 資料夾存在
os.makedirs('output', exist_ok=True)

# 定義實驗規模 (Size Grid)
SIZE_GRID = {
    'mimic3c': [100, 500, 6000]
}

## 1. 資料載入與不平衡子集生成 (方案 A)

In [8]:
def load_mimic3c():
    """
    載入資料並執行標籤洩漏排除與輕量前處理 (切換為方案 B)
    """
    print("Loading data using Scheme B (從原始資料重新前處理)...")
    # 讀取 Kaggle 原始資料
    df = pd.read_csv('data/mimic3c.csv')
    
    target_col = 'ExpiredHospital'
    
    # 1. ⚠️ 嚴格排除標籤洩漏特徵與無用 ID
    # hadm_id 是流水號，不能拿來訓練，否則 One-Hot 後會暴增幾萬個欄位
    leakage_cols = ['LOSdays', 'LOSgroupNum', 'AdmitDiagnosis', 'hadm_id']
    cols_to_drop = [c for c in leakage_cols if c in df.columns]
    df = df.drop(columns=cols_to_drop)
    print(f"已排除標籤洩漏與 ID 欄位: {cols_to_drop}")
    
    # 2. 輕量前處理：處理缺失值
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    if target_col in numeric_cols:
        numeric_cols.remove(target_col) # 確保不處理到目標變數
        
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # 數值補中位數，類別補眾數
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())
    for col in categorical_cols:
        df[col] = df[col].fillna(df[col].mode()[0])
        
    # 3. 類別變數編碼 (One-Hot Encoding)
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    
    # 4. 提取 X, y (將 X 轉為 numpy array 以加速後續 840 次的迴圈訓練)
    y = df[target_col].astype(int).values
    X = df.drop(columns=[target_col]).values
    
    print(f"X shape: {X.shape}")
    print(f"y=1 (死亡) 原始比例: {(y.sum() / len(y)):.4f}")
    return X, y

# 重新測試執行
X_full, y_full = load_mimic3c()
X_sub, y_sub = make_imbalanced_subset(X_full, y_full, n_total=500)
print(f"抽樣測試 n=500 -> 少數類比例: {y_sub.sum() / len(y_sub):.2f}")

Loading data using Scheme B (從原始資料重新前處理)...
已排除標籤洩漏與 ID 欄位: ['LOSdays', 'LOSgroupNum', 'AdmitDiagnosis', 'hadm_id']
X shape: (58976, 1372)
y=1 (死亡) 原始比例: 0.0993
抽樣測試 n=500 -> 少數類比例: 0.10


## 2. 核心訓練迴圈 (20-seed Experiment)

In [9]:
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.base import clone

# 載入所有需要的模型
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# 1. 定義七個模型 (⚠️ 嚴格固定有隨機性的模型的 random_state=0)
seven_models = {
    'DecisionTree': DecisionTreeClassifier(max_depth=15, random_state=0),
    'KNN':          KNeighborsClassifier(n_neighbors=5), # KNN 依賴距離，無隨機初始化
    'LogisticReg':  LogisticRegression(max_iter=1000, random_state=0),
    'SVM':          SVC(kernel='rbf', gamma='scale', random_state=0),
    'MLP':          MLPClassifier(hidden_layer_sizes=(128, 128), 
                                  max_iter=200, early_stopping=True, 
                                  random_state=0),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=0),
    'NaiveBayes':   GaussianNB() # NB 是基於機率分佈的封閉解，無隨機性
}

def fit_and_score(model_template, X_train, y_train, X_test, y_test):
    """
    負責單次訓練並回傳 Macro F1 分數
    """
    model = clone(model_template) # ⚠️ 必須 clone，否則模型會記住上一輪的權重！
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    # 使用 macro F1，確保在極度不平衡下兩類都被公平考量
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    return macro_f1

def run_one_dataset(X, y, n, n_repeats=20):
    """
    針對單一 size 的資料，執行 20-seed 雙策略比較
    """
    print(f"\n--- 執行 Size n={n} 實驗 ---")
    rows = []
    
    # 執行 20 次 seed
    for seed in range(n_repeats):
        if seed % 5 == 0:
            print(f"  > 處理 Seed {seed}/{n_repeats}...")
            
        for strategy in ['Stratified', 'Random']:
            # 1. Train/Test Split
            stratify_param = y if strategy == 'Stratified' else None
            
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.2, random_state=seed, stratify=stratify_param
            )
            
            # 計算測試集中的少數類比例 (監控 Random Split 有多常出現極端值)
            test_minority_ratio = y_te.sum() / len(y_te) if len(y_te) > 0 else 0
            
            # 2. 標準化 (⚠️ 必須 fit on train, transform on test 避免洩漏)
            scaler = StandardScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_te_scaled = scaler.transform(X_te)
            
            # 3. 訓練七個模型
            for model_name, model_template in seven_models.items():
                f1 = fit_and_score(model_template, X_tr_scaled, y_tr, X_te_scaled, y_te)
                
                rows.append({
                    'seed': seed,
                    'strategy': strategy,
                    'model': model_name,
                    'n_samples': n,
                    'macro_f1': f1,
                    'test_minority_ratio': test_minority_ratio
                })
                
    return pd.DataFrame(rows)

# =============== 核心主流程 ===============
print("開始執行核心實驗，這將會花費數分鐘時間，請耐心等候...")
start_time = time.time()

all_dfs = []
# 確保我們已經從 Step 2 拿到了乾淨的 X_full, y_full
for n in SIZE_GRID['mimic3c']:
    # 每次都用 seed=42 抽出一個固定不平衡的母體子集
    X_sub, y_sub = make_imbalanced_subset(X_full, y_full, n, imbalance_ratio=0.1, random_state=42)
    
    # 開始跑迴圈
    df_result = run_one_dataset(X_sub, y_sub, n, n_repeats=20)
    df_result['dataset'] = 'mimic3c'
    all_dfs.append(df_result)

# 合併所有結果並存檔
final_results_df = pd.concat(all_dfs, ignore_index=True)
final_results_df.to_csv('output/mimic3c_results.csv', index=False)

end_time = time.time()
print(f"\n✅ 實驗完成！總計產生 {len(final_results_df)} 筆紀錄，耗時 {(end_time - start_time):.2f} 秒。")
print(f"結果已儲存至 'output/mimic3c_results.csv'")

開始執行核心實驗，這將會花費數分鐘時間，請耐心等候...

--- 執行 Size n=100 實驗 ---
  > 處理 Seed 0/20...
  > 處理 Seed 5/20...
  > 處理 Seed 10/20...
  > 處理 Seed 15/20...

--- 執行 Size n=500 實驗 ---
  > 處理 Seed 0/20...
  > 處理 Seed 5/20...
  > 處理 Seed 10/20...
  > 處理 Seed 15/20...

--- 執行 Size n=6000 實驗 ---
  > 處理 Seed 0/20...
  > 處理 Seed 5/20...
  > 處理 Seed 10/20...
  > 處理 Seed 15/20...

✅ 實驗完成！總計產生 840 筆紀錄，耗時 423.01 秒。
結果已儲存至 'output/mimic3c_results.csv'


## 3. 統計分析與視覺化 (Statistical Analysis & Visualization)

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel

# 讀取剛才跑完的實驗結果
df_results = pd.read_csv('output/mimic3c_results.csv')

# ==========================================
# (a) 繪製 Boxplot 
# ==========================================
def plot_boxplot(df_results, n):
    df_n = df_results[df_results['n_samples'] == n]
    
    plt.figure(figsize=(12, 6))
    sns.boxplot(x='model', y='macro_f1', hue='strategy', data=df_n, palette='Set2')
    plt.title(f'Macro F1 Distribution by Strategy and Model (n={n})')
    plt.ylabel('Macro F1 Score')
    plt.xlabel('Model')
    plt.xticks(rotation=45)
    plt.legend(title='Strategy', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    
    # 存檔
    plt.savefig(f'output/mimic3c_boxplot_n{n}.png')
    plt.close() # 關閉畫布避免 memory leak

for n in [100, 500, 6000]:
    plot_boxplot(df_results, n)
print("✅ Boxplot 繪製完成並已存檔！")

# ==========================================
# (b) 統計分析: Paired t-test 與 Significance Table
# ==========================================
def paired_significance_table(df_results):
    rows = []
    # 遍歷 3 個 size
    for n in df_results['n_samples'].unique():
        df_n = df_results[df_results['n_samples'] == n]
        
        # 遍歷 7 個模型
        for model in df_n['model'].unique():
            df_m = df_n[df_n['model'] == model]
            
            # 確保 seed 順序一致才能做 paired t-test
            stratified_f1 = df_m[df_m['strategy'] == 'Stratified'].sort_values('seed')['macro_f1'].values
            random_f1 = df_m[df_m['strategy'] == 'Random'].sort_values('seed')['macro_f1'].values
            
            # 計算平均差與 t-test
            mean_diff = np.mean(stratified_f1 - random_f1)
            t_value, p_value = ttest_rel(stratified_f1, random_f1)
            
            # 判定結果
            if p_value < 0.05:
                better = 'Stratified' if mean_diff > 0 else 'Random'
            else:
                better = 'n.s.' # 不顯著 (not significant)
                
            rows.append({
                'n': n,
                'model': model,
                't_value': round(t_value, 4),
                'p_value': p_value, # p-value 通常保留科學記號或多位數
                'mean_diff': round(mean_diff, 4),
                'better': better
            })
            
    df_sig = pd.DataFrame(rows)
    # 輸出完整版 CSV
    df_sig.to_csv('output/significance_full.csv', index=False)
    
    # 篩選 p < 0.05 並輸出 Markdown 表格
    df_sig_filtered = df_sig[df_sig['p_value'] < 0.05]
    with open('output/significance_table.md', 'w') as f:
        f.write(df_sig_filtered.to_markdown(index=False))
        
    return df_sig

sig_table = paired_significance_table(df_results)
print("✅ Paired t-test 檢定完成，表格已匯出！")

# ==========================================
# (c) Ceiling Effect Table (針對 n=500)
# ==========================================
def ceiling_effect_table(df_results, n_focus=500):
    df_focus = df_results[df_results['n_samples'] == n_focus]
    rows = []
    
    for model in df_focus['model'].unique():
        df_m = df_focus[df_focus['model'] == model]
        
        # mean_f1: 兩策略 40 筆的平均
        mean_f1 = df_m['macro_f1'].mean()
        
        # diff_std: 20 個 seed 的成對差值標準差
        stratified_f1 = df_m[df_m['strategy'] == 'Stratified'].sort_values('seed')['macro_f1'].values
        random_f1 = df_m[df_m['strategy'] == 'Random'].sort_values('seed')['macro_f1'].values
        diff_std = np.std(stratified_f1 - random_f1, ddof=1) # 樣本標準差
        
        rows.append({
            'model': model,
            'mimic3c_mean_f1': round(mean_f1, 4),
            'mimic3c_diff_std': round(diff_std, 6)
        })
        
    df_ceiling = pd.DataFrame(rows)
    with open('output/ceiling_effect_table.md', 'w') as f:
        f.write(df_ceiling.to_markdown(index=False))
        
ceiling_effect_table(df_results)
print("✅ Ceiling Effect 表格已匯出！")

# ==========================================
# (d) F1 STD Ratio Table (針對 n=100)
# ==========================================
def std_ratio_table(df_results, n_focus=100):
    df_focus = df_results[df_results['n_samples'] == n_focus]
    rows = []
    
    for model in df_focus['model'].unique():
        df_m = df_focus[df_focus['model'] == model]
        
        std_stratified = df_m[df_m['strategy'] == 'Stratified']['macro_f1'].std()
        std_random = df_m[df_m['strategy'] == 'Random']['macro_f1'].std()
        
        ratio = std_stratified / std_random if std_random != 0 else np.nan
        
        rows.append({
            'model': model,
            'mimic3c_ratio': round(ratio, 4)
        })
        
    df_ratio = pd.DataFrame(rows)
    with open('output/std_ratio_table.md', 'w') as f:
        f.write(df_ratio.to_markdown(index=False))

std_ratio_table(df_results)
print("✅ STD Ratio 表格已匯出！")

✅ Boxplot 繪製完成並已存檔！
✅ Paired t-test 檢定完成，表格已匯出！
✅ Ceiling Effect 表格已匯出！
✅ STD Ratio 表格已匯出！


In [11]:
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import sklearn
import scipy

print(f"pandas=={pd.__version__}")
print(f"numpy=={np.__version__}")
print(f"matplotlib=={matplotlib.__version__}")
print(f"seaborn=={sns.__version__}")
print(f"scikit-learn=={sklearn.__version__}")
print(f"scipy=={scipy.__version__}")

pandas==2.2.3
numpy==2.1.3
matplotlib==3.10.1
seaborn==0.13.2
scikit-learn==1.6.1
scipy==1.15.2
